# Milan Crashes Processing

This notebook cleans the Milano crash raw datasets and writes mirror cleaned files to data/processed.

Reference: ../data/raw/raw.md

In [11]:
from pathlib import Path

import pandas as pd

pd.options.display.max_columns = 200
pd.options.display.float_format = "{:.3f}".format

candidate_roots = [Path.cwd(), Path.cwd().parent]
project_root = None
for root in candidate_roots:
    if (root / "data" / "raw").exists() and (root / "data" / "processed").exists():
        project_root = root
        break

if project_root is None:
    raise FileNotFoundError("Could not resolve the project root with data/raw and data/processed.")

RAW_DIR = project_root / "data" / "raw"
PROCESSED_DIR = project_root / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = {
    "milan_crashes_monthly": {
        "file": RAW_DIR / "milan_crashes_monthly.csv",
        "id_cols": ["Anno", "Mese"],
    },
    "milan_crashes_monthly_city_ring": {
        "file": RAW_DIR / "milan_crashes_monthly_city_ring.csv",
        "id_cols": ["Anno", "Mese", "Cerchia"],
    },
    "milan_crashes_by_nature": {
        "file": RAW_DIR / "milan_crashes_by_nature.csv",
        "id_cols": ["Anno", "Mese", "NaturaIncidente"],
    },
    "milan_crashes_by_vehicles": {
        "file": RAW_DIR / "milan_crashes_by_vehicles.csv",
        "id_cols": ["Anno", "Mese"],
    },
    "milan_crashes_by_zone": {
        "file": RAW_DIR / "milan_crashes_by_zone.csv",
        "id_cols": ["Anno", "Mese", "Municipio"],
    },
}


def load_raw_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep=";", encoding="utf-8-sig")

In [12]:
# Cleaning function each dataset can be run through
def clean_common(df: pd.DataFrame, id_cols: list[str]) -> pd.DataFrame:
    out = df.copy()
    out.columns = out.columns.astype(str).str.strip()

    id_cols_set = set(id_cols)
    special_cols = {"Anno", "Mese"}

    for col in out.columns:
        # Convert "Anno" and "Mese" to numeric
        if col in special_cols:
            out[col] = pd.to_numeric(out[col], errors="coerce").astype("Int64")
            continue

        # Check if the column is string/object
        is_str_or_obj = pd.api.types.is_string_dtype(
            out[col]
        ) or pd.api.types.is_object_dtype(out[col])

        # If ID is string, set NA
        if col in id_cols_set:
            if is_str_or_obj:
                out[col] = out[col].astype("string").str.strip().replace("", pd.NA)
            continue

        # All other string columns, set NA + attempt numeric conversion
        if is_str_or_obj:
            s_clean = out[col].astype("string").str.strip().replace("", pd.NA)

            converted = pd.to_numeric(s_clean, errors="coerce")

            if converted.notna().mean() > 0.8:
                out[col] = converted
            else:
                out[col] = s_clean
    
    # Create "month_start" datetime column if "Anno" and "Mese" are present
    if special_cols.issubset(out.columns):
        out["month_start"] = pd.to_datetime(
            {
                "year": out["Anno"],
                "month": out["Mese"],
                "day": 1,
            },
            errors="coerce",
        )

    return out

In [13]:
# Replace spaces with a single space, remove spaces around dashes
def normalize_natura_incidente(series: pd.Series) -> pd.Series:
    out = series.astype("string").str.strip()
    out = out.str.replace(r"\s+", " ", regex=True)
    out = out.str.replace(r"\s*-\s*", "-", regex=True)
    return out

In [14]:
def clean_dataset(name: str, cfg: dict) -> tuple[pd.DataFrame, dict]:
    raw = load_raw_csv(cfg["file"])
    before_rows = len(raw)

    cleaned = clean_common(raw, cfg["id_cols"])

    # Keep unknown IDs as NA for consistency across datasets.
    if name == "milan_crashes_by_zone":
        cleaned["Municipio"] = pd.to_numeric(
            cleaned["Municipio"], errors="coerce"
        ).astype("Int64")

    if name == "milan_crashes_by_nature":
        cleaned["NaturaIncidente"] = normalize_natura_incidente(
            cleaned["NaturaIncidente"]
        )

    numeric_cols = [
        c
        for c in cleaned.columns
        if c not in cfg["id_cols"]
        and c not in {"month_start"}
        and pd.api.types.is_numeric_dtype(cleaned[c])
    ]
    for col in numeric_cols:
        cleaned[col] = cleaned[col].round().astype("Int64")

    cleaned = (
        cleaned.drop_duplicates()
        .sort_values(
            [c for c in ["Anno", "Mese"] if c in cleaned.columns]
            + [c for c in cfg["id_cols"] if c not in {"Anno", "Mese"}],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    qc = {
        "rows_before": before_rows,
        "rows_after": len(cleaned),
        "duplicates_removed": before_rows - len(cleaned),
        "missing_cells": int(cleaned.isna().sum().sum()),
    }
    return cleaned, qc

In [15]:
cleaned_data = {}
qc_rows = []

for name, cfg in DATASETS.items():
    cleaned_df, qc = clean_dataset(name, cfg)
    cleaned_data[name] = cleaned_df

    invalid_month_rows = (
        int((~cleaned_df["Mese"].between(1, 12)).fillna(True).sum())
        if "Mese" in cleaned_df.columns
        else 0
    )

    num_df = cleaned_df.select_dtypes(include=["number"])
    negative_numeric_cells = int((num_df < 0).sum().sum()) if not num_df.empty else 0

    qc_rows.append(
        {
            "dataset": name,
            "rows_before": qc["rows_before"],
            "rows_after": qc["rows_after"],
            "duplicates_removed": qc["duplicates_removed"],
            "invalid_month_rows": invalid_month_rows,
            "negative_numeric_cells": negative_numeric_cells,
            "missing_cells": int(cleaned_df.isna().sum().sum()),
        }
    )

qc_scan = pd.DataFrame(qc_rows).sort_values("dataset").reset_index(drop=True)
qc_scan

,dataset,rows_before,rows_after,duplicates_removed,invalid_month_rows,negative_numeric_cells,missing_cells
0,milan_crashes_by_nature,2590,2590,0,0,0,0
1,milan_crashes_by_vehicles,288,288,0,0,0,174
2,milan_crashes_by_zone,2880,2880,0,0,0,288
3,milan_crashes_monthly,288,288,0,0,0,3
4,milan_crashes_monthly_city_ring,1440,1440,0,0,0,0


## Additional Validation Gate

This gate fails fast when core temporal integrity checks are broken (invalid months or malformed month_start rows).

In [18]:
validation_rows = []
for dataset_name, frame in cleaned_data.items():
    invalid_month = int((~frame["Mese"].between(1, 12)).fillna(True).sum()) if "Mese" in frame.columns else 0
    invalid_month_start = int(frame["month_start"].isna().sum()) if "month_start" in frame.columns else 0
    validation_rows.append(
        {
            "dataset": dataset_name,
            "invalid_month_rows": invalid_month,
            "invalid_month_start_rows": invalid_month_start,
        }
    )

validation_df = pd.DataFrame(validation_rows).sort_values("dataset").reset_index(drop=True)
display(validation_df)

if (validation_df[["invalid_month_rows", "invalid_month_start_rows"]].sum().sum()) > 0:
    raise ValueError("Validation gate failed: found invalid month or month_start values in cleaned crash datasets.")

print("Validation gate passed: all cleaned crash datasets have valid month and month_start values.")

,dataset,invalid_month_rows,invalid_month_start_rows
0,milan_crashes_by_nature,0,0
1,milan_crashes_by_vehicles,0,0
2,milan_crashes_by_zone,0,0
3,milan_crashes_monthly,0,0
4,milan_crashes_monthly_city_ring,0,0


Validation gate passed: all cleaned crash datasets have valid month and month_start values.


In [19]:
for name, df in cleaned_data.items():
    out_file = PROCESSED_DIR / f"{name}_cleaned.csv"
    df.to_csv(out_file, index=False)

print(f"Saved {len(cleaned_data)} cleaned crash datasets in: {PROCESSED_DIR.resolve()}")

Saved 5 cleaned crash datasets in: /home/iuseppe_ares/projects/APPSTAT/MilanCrash/data/processed
